In [150]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [151]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [152]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [153]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [305]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | dt.shape: {dt.shape}, t.shape:{t.grad.shape}')
  print(f'{s:15s} | dt[0]: {dt[0]}, t[0]: {t.grad[0]}')
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [301]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [302]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [306]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
logit_maxes_ind = logits.max(1, keepdim=True).indices
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3416, grad_fn=<NegBackward0>)

In [307]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1/n
dprobs = dlogprobs*(1/probs)
dcounts_sum_inv_exp = counts*dprobs
dcounts_sum_inv = dcounts_sum_inv_exp.sum(1, keepdim=True)
dcounts_sum = -(counts_sum**-2)*dcounts_sum_inv
dcounts = counts_sum_inv*dprobs + torch.ones_like(counts)*dcounts_sum
dnorm_logits = counts*dcounts
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
dlogits = dnorm_logits.clone()
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1])*dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0, keepdim=True)
dhpreact = dh*(1-h**2)
dbngain = (dhpreact*bnraw).sum(0, keepdim=True)
dbnbias = dhpreact.clone().sum(0, keepdim=True)
dbnraw = dhpreact*bngain
dbnvar_inv = (dbnraw*bndiff).sum(0, keepdim=True)
dbndiff = dbnraw*bnvar_inv
dbnvar = -0.5*dbnvar_inv*((bnvar + 1e-5)**-1.5)
dbndiff2 = (1/(n-1))*(torch.ones_like(bndiff2)*dbnvar)
dbndiff += 2*bndiff*dbndiff2
dbnmeani = -1*dbndiff.sum(0, keepdim=True)
dhprebn = dbndiff.clone()
dhprebn += (1/n)*(torch.ones_like(hprebn)*dbnmeani)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn 
db1 = dhprebn.sum(0, keepdim=True)
demb = dembcat.view(emb.shape[0], emb.shape[1], -1)
dC = torch.zeros_like(C)
for r_idx, row in enumerate(Xb):
    for c_idx, value in enumerate(row):
        # print(dC[value].shape, demb[r_idx][c_idx].shape)
        dC[value] += demb[r_idx][c_idx]

# print(torch.ones_like(bndiff2).shape)
# print(dbnvar.shape)
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | dt.shape: torch.Size([32, 27]), t.shape:torch.Size([32, 27])
logprobs        | dt[0]: tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -0.0312,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000]), t[0]: tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -0.0312,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000])
logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | dt.shape: torch.Size([32, 27]), t.shape:torch.Size([32, 27])
probs           | dt[0]: tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -1.6639,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
      

In [285]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.456361770629883 diff: -2.384185791015625e-07


In [316]:
# backward pass

# -----------------
# YOUR CODE HERE :)
# dlogits = torch.zeros_like(logits) # TODO. my solution is 3 lines
dlogits = F.softmax(logits, dim=1)
dlogits[range(n), Yb] -= 1
dlogits /= n
# for i in range(logits.shape[0]):
#     print(f'i: {i}, Yb[i]: {Yb[i]}')
#     print(f'logits.grad[i]: {logits.grad[i]}')
#     print(f'dlogits[i]: {dlogits[i]}')
# -----------------

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | dt.shape: torch.Size([32, 27]), t.shape:torch.Size([32, 27])
logits          | dt[0]: tensor([ 0.0023,  0.0026,  0.0006,  0.0015,  0.0006,  0.0026,  0.0008,  0.0012,
        -0.0307,  0.0010,  0.0012,  0.0012,  0.0012,  0.0009,  0.0011,  0.0005,
         0.0003,  0.0006,  0.0005,  0.0018,  0.0017,  0.0007,  0.0008,  0.0020,
         0.0018,  0.0008,  0.0007], grad_fn=<SelectBackward0>), t[0]: tensor([ 0.0023,  0.0026,  0.0006,  0.0015,  0.0006,  0.0026,  0.0008,  0.0012,
        -0.0307,  0.0010,  0.0012,  0.0012,  0.0012,  0.0009,  0.0011,  0.0005,
         0.0003,  0.0006,  0.0005,  0.0018,  0.0017,  0.0007,  0.0008,  0.0020,
         0.0018,  0.0008,  0.0007])
logits          | exact: False | approximate: True  | maxdiff: 6.51925802230835e-09


In [13]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(7.1526e-07, grad_fn=<MaxBackward1>)


In [328]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
print(bngain.shape)
dhprebn = bngain*bnvar_inv/n*(n*dhpreact - dhpreact.sum(0) - (n/(n-1))*bnraw*(dhpreact*bnraw).sum(0))
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

torch.Size([1, 64])
hprebn          | dt.shape: torch.Size([32, 64]), t.shape:torch.Size([32, 64])
hprebn          | dt[0]: tensor([-3.3140e-04, -6.5023e-04,  1.1324e-03,  6.3152e-04, -1.1331e-03,
        -2.9402e-04,  2.0485e-03, -7.2168e-04,  3.1304e-04, -3.9707e-04,
        -1.4724e-03,  7.1713e-04,  1.5069e-03,  1.5261e-03, -1.1182e-03,
        -2.0224e-03,  3.4135e-04,  4.6498e-05,  1.2111e-03,  7.2378e-04,
         1.9495e-03,  1.5982e-03,  1.0370e-03,  2.4709e-04,  8.3440e-04,
         2.6437e-04, -3.6841e-04, -4.3800e-03, -2.6097e-04,  3.2509e-03,
         2.9607e-04, -3.0542e-03, -5.5095e-05,  4.8614e-04, -2.4480e-04,
        -6.1638e-04,  7.7313e-04,  7.9929e-04, -7.8561e-04, -6.3122e-04,
        -4.1364e-04, -4.9703e-04, -1.7622e-03, -1.2143e-03, -1.8090e-03,
         5.6718e-05,  5.8592e-04, -1.3228e-03,  5.0103e-04,  1.4892e-03,
        -1.2278e-03, -2.0406e-03, -3.9879e-04,  1.1023e-04, -3.0986e-04,
        -4.3017e-04, -8.4622e-04,  2.7299e-03, -2.5101e-04,  3.9521e-03,
